In [42]:
# =============================================================================
#  SHROOM-visions 2026 — text-only span tagger
#  XLM-R token classification -> per-character hallucination probability
#
#  Kaggle: T4, ~20 min end to end. Expects the data already extracted at
#  /kaggle/working/distrib/ (as in the earlier notebook).
#
#  Produces: dev scores + predictions_{lang}.jsonl.
# =============================================================================

In [43]:
ls /kaggle/working/distrib

shroom-vision.test.en.unlabeled.jsonl  shroom-vision.train.en.labeled.jsonl
shroom-vision.test.fr.unlabeled.jsonl  shroom-vision.train.fr.labeled.jsonl
shroom-vision.test.it.unlabeled.jsonl  shroom-vision.train.it.labeled.jsonl
shroom-vision.test.zh.unlabeled.jsonl  shroom-vision.train.zh.labeled.jsonl


In [44]:
# ── Cell 0: fetch data if not already present ────────────────────────────────
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
if not os.path.exists("/kaggle/working/distrib"):
    !wget -q https://a3s.fi/mickusti-2007780-pub/shroom-visions-data.zip -O /tmp/data.zip
    !unzip -oq /tmp/data.zip -d /kaggle/working/
print(sorted(os.listdir("/kaggle/working/distrib")))

['shroom-vision.test.en.unlabeled.jsonl', 'shroom-vision.test.fr.unlabeled.jsonl', 'shroom-vision.test.it.unlabeled.jsonl', 'shroom-vision.test.zh.unlabeled.jsonl', 'shroom-vision.train.en.labeled.jsonl', 'shroom-vision.train.fr.labeled.jsonl', 'shroom-vision.train.it.labeled.jsonl', 'shroom-vision.train.zh.labeled.jsonl']


In [45]:
# ── Cell 1: setup ────────────────────────────────────────────────────────────
import json, os, random, pathlib
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from scipy.stats import spearmanr

DISTRIB    = "/kaggle/working/distrib"
OUT_DIR    = "/kaggle/working"
MODEL_ID   = "xlm-roberta-large"
LANGS      = ["en", "fr", "it", "zh"]
CATEGORIES = ["invention", "mischaracterization", "OCR", "miscounting", "other"]
MAX_LEN    = 256
BATCH      = 8
EPOCHS     = 5
LR         = 1e-5
SEED       = 13

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


In [46]:
# ── Cell 2: official scoring functions ───────────────────────────────────────
# Copied verbatim from the organizers' scorer.py so dev numbers match the
# leaderboard exactly. Do not edit.

def score_cor(ref_dict, pred_dict, label_filtered_=None):
    assert ref_dict['id'] == pred_dict['id']
    ref_vec = [0.] * ref_dict['text_len']
    pred_vec = [0.] * ref_dict['text_len']
    ref_labels = (ref_dict['labels'] if label_filtered_ is None
                  else [s for s in ref_dict['labels'] if s['label'] == label_filtered_])
    pred_labels = (pred_dict['labels'] if label_filtered_ is None
                   else [s for s in pred_dict['labels'] if s['label'] == label_filtered_])
    for span in ref_labels:
        for idx in range(span['start'], span['end']):
            ref_vec[idx] += span['prob']
    for span in pred_labels:
        for idx in range(span['start'], span['end']):
            pred_vec[idx] = span['prob']
    ref_cmps = {round(f, 8) for f in ref_vec}
    pred_cmps = {round(f, 8) for f in pred_vec}
    if len(pred_cmps) == 1 or len(ref_cmps) == 1:
        if len(pred_cmps) != len(ref_cmps):
            return 0.0
        if ref_cmps == {0.0}:
            return float(pred_cmps == {0.0})
        return float(pred_cmps != {0.0})
    return spearmanr(ref_vec, pred_vec).correlation


def score_cor_lbl(ref_dict, pred_dict):
    all_labels = {s['label'] for d in [ref_dict, pred_dict] for s in d['labels']}
    if all_labels:
        return sum(score_cor(ref_dict, pred_dict, label_filtered_=l)
                   for l in all_labels) / len(all_labels)
    return 1.0


def score_iou(ref_dict, pred_dict):
    assert ref_dict['id'] == pred_dict['id']
    ref_i = {i for s in ref_dict['labels'] for i in range(s['start'], s['end'])}
    pred_i = {i for s in pred_dict['labels'] for i in range(s['start'], s['end'])}
    if not pred_i and not ref_i:
        return 1.
    return len(ref_i & pred_i) / len(ref_i | pred_i)


def evaluate(refs, preds):
    refs = sorted(refs, key=lambda r: r['id'])
    preds = sorted(preds, key=lambda r: r['id'])
    assert [r['id'] for r in refs] == [p['id'] for p in preds]
    return {
        'Cor':     float(np.mean([score_cor(r, p)     for r, p in zip(refs, preds)])),
        'Cor_lbl': float(np.mean([score_cor_lbl(r, p) for r, p in zip(refs, preds)])),
        'IoU':     float(np.mean([score_iou(r, p)     for r, p in zip(refs, preds)])),
    }

In [47]:
# ── Cell 3: data ─────────────────────────────────────────────────────────────
def load(lang, split_name):
    kind = "labeled" if split_name == "train" else "unlabeled"
    path = f"{DISTRIB}/shroom-vision.{split_name}.{lang}.{kind}.jsonl"
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]


def split_dev(rows, fraction=0.2, seed=SEED):
    """Stratified split preserving the clean / hallucinated ratio."""
    clean = [r for r in rows if not r["labels"]]
    dirty = [r for r in rows if r["labels"]]
    rng = random.Random(seed)
    rng.shuffle(clean); rng.shuffle(dirty)
    nc, nd = int(len(clean) * fraction), int(len(dirty) * fraction)
    dev = clean[:nc] + dirty[:nd]
    train = clean[nc:] + dirty[nd:]
    rng.shuffle(dev); rng.shuffle(train)
    return train, dev


def char_targets(row):
    n = len(row["response"])
    prob = np.zeros(n, dtype=np.float32)
    cat = np.full(n, -1, dtype=np.int64)
    for span in row.get("labels", []):
        for i in range(span["start"], min(span["end"], n)):
            if span["prob"] >= prob[i]:
                prob[i] = span["prob"]
                cat[i] = CATEGORIES.index(span["label"])
    return prob, cat


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def encode(row):
    """Tokenize response (segment 0) with the prompt as context (segment 1)."""
    enc = tokenizer(row["response"], text_pair=row["prompt"],
                    return_offsets_mapping=True, truncation="only_first",
                    max_length=MAX_LEN)
    seq_ids = enc.sequence_ids()
    keep = [i for i, (a, b) in enumerate(enc["offset_mapping"])
            if b > a and seq_ids[i] == 0]
    return enc, keep


class SpanData(Dataset):
    def __init__(self, rows, labeled=True):
        self.rows, self.labeled = rows, labeled
        # Tokenize once up front. Doing it per access pins the CPU at 100% and
        # starves the GPU, since the loader runs in the main process.
        self.cache = [encode(r) for r in rows]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        enc, keep = self.cache[i]
        item = {
            "input_ids": torch.tensor(enc["input_ids"]),
            "attention_mask": torch.tensor(enc["attention_mask"]),
            "keep": torch.tensor(keep, dtype=torch.long),
        }
        if self.labeled:
            prob, cat = char_targets(row)
            offs = [enc["offset_mapping"][i] for i in keep]
            # per token: max character probability, and the category at the peak
            tp = [float(prob[a:b].max()) if b > a else 0.0 for a, b in offs]
            tc = []
            for a, b in offs:
                seg = cat[a:b]
                seg = seg[seg >= 0]
                tc.append(int(np.bincount(seg).argmax()) if len(seg) else -100)
            item["tok_prob"] = torch.tensor(tp, dtype=torch.float)
            item["tok_cat"] = torch.tensor(tc, dtype=torch.long)
        return item


def collate(batch):
    pad = tokenizer.pad_token_id
    maxlen = max(len(b["input_ids"]) for b in batch)
    maxkeep = max(len(b["keep"]) for b in batch)
    out = {
        "input_ids": torch.full((len(batch), maxlen), pad, dtype=torch.long),
        "attention_mask": torch.zeros((len(batch), maxlen), dtype=torch.long),
        "keep": torch.zeros((len(batch), maxkeep), dtype=torch.long),
        "keep_mask": torch.zeros((len(batch), maxkeep), dtype=torch.bool),
    }
    has_labels = "tok_prob" in batch[0]
    if has_labels:
        out["tok_prob"] = torch.zeros((len(batch), maxkeep))
        out["tok_cat"] = torch.full((len(batch), maxkeep), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        L, K = len(b["input_ids"]), len(b["keep"])
        out["input_ids"][i, :L] = b["input_ids"]
        out["attention_mask"][i, :L] = b["attention_mask"]
        out["keep"][i, :K] = b["keep"]
        out["keep_mask"][i, :K] = True
        if has_labels:
            out["tok_prob"][i, :K] = b["tok_prob"]
            out["tok_cat"][i, :K] = b["tok_cat"]
    return out

In [48]:
# ── Cell 4: model ────────────────────────────────────────────────────────────
class SpanTagger(nn.Module):
    """One shared encoder, two token-level heads.

    span head  -> is this token hallucinated        (drives Cor)
    cat  head  -> which of the five categories      (drives Cor_lbl only)
    """

    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_ID)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.span_head = nn.Linear(h, 1)
        self.cat_head = nn.Linear(h, len(CATEGORIES))

    def forward(self, input_ids, attention_mask, keep, keep_mask):
        hidden = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask).last_hidden_state
        idx = keep.unsqueeze(-1).expand(-1, -1, hidden.size(-1))
        tok = self.dropout(torch.gather(hidden, 1, idx))
        return self.span_head(tok).squeeze(-1), self.cat_head(tok)


def run_epoch(model, loader, optimizer=None, scheduler=None, log_every=50):
    train = optimizer is not None
    model.train() if train else model.eval()
    bce = nn.BCEWithLogitsLoss(reduction="none")
    ce = nn.CrossEntropyLoss(ignore_index=-100)
    total, nb = 0.0, 0
    for step, batch in enumerate(loader, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.set_grad_enabled(train):
            span_logit, cat_logit = model(batch["input_ids"], batch["attention_mask"],
                                          batch["keep"], batch["keep_mask"])
            m = batch["keep_mask"]
            # soft targets: BCE against the annotator probability itself
            l_span = (bce(span_logit, batch["tok_prob"]) * m).sum() / m.sum().clamp(min=1)
            l_cat = ce(cat_logit.reshape(-1, len(CATEGORIES)), batch["tok_cat"].reshape(-1))
            loss = l_span + 0.5 * l_cat
        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total += loss.item(); nb += 1
        if train and step % log_every == 0:
            print(f"  step {step}/{len(loader)}  loss {total / nb:.4f}", flush=True)
    return total / max(nb, 1)

In [49]:
# ── Cell 5: inference ────────────────────────────────────────────────────────
@torch.no_grad()
def predict_char_probs(model, rows):
    """Return per-character probability and category arrays for each row."""
    model.eval()
    dataset = SpanData(rows, labeled=False)
    loader = DataLoader(dataset, batch_size=BATCH, shuffle=False, collate_fn=collate)
    results, cursor = [], 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        span_logit, cat_logit = model(batch["input_ids"], batch["attention_mask"],
                                      batch["keep"], batch["keep_mask"])
        probs = torch.sigmoid(span_logit).cpu().numpy()
        cats = cat_logit.argmax(-1).cpu().numpy()
        for i in range(len(probs)):
            row = rows[cursor]
            enc, keep = dataset.cache[cursor]
            cursor += 1
            offs = [enc["offset_mapping"][k] for k in keep]
            n = len(row["response"])
            cp = np.zeros(n, dtype=np.float32)
            cc = np.zeros(n, dtype=np.int64)
            for j, (a, b) in enumerate(offs):
                cp[a:min(b, n)] = probs[i][j]
                cc[a:min(b, n)] = cats[i][j]
            results.append((cp, cc))
    return results


def to_spans(char_prob, char_cat, threshold):
    """Contiguous above-threshold runs, split where the category changes."""
    spans, n, i = [], len(char_prob), 0
    while i < n:
        if char_prob[i] < threshold:
            i += 1; continue
        j, c = i, char_cat[i]
        while j < n and char_prob[j] >= threshold and char_cat[j] == c:
            j += 1
        spans.append({"start": int(i), "end": int(j),
                      "prob": float(round(float(np.mean(char_prob[i:j])), 6)),
                      "label": CATEGORIES[int(c)]})
        i = j
    return spans


def build_preds(rows, char_preds, threshold):
    return [{"id": r["id"], "labels": to_spans(cp, cc, threshold)}
            for r, (cp, cc) in zip(rows, char_preds)]


def tune_threshold(dev_rows, char_preds):
    """Pick the threshold maximizing Cor on dev. Cor is the reported metric."""
    refs = [{"id": r["id"], "labels": r["labels"], "text_len": len(r["response"])}
            for r in dev_rows]
    best = (None, -1, None)
    for t in np.arange(0.10, 0.91, 0.05):
        s = evaluate(refs, build_preds(dev_rows, char_preds, float(t)))
        if s["Cor"] > best[1]:
            best = (float(t), s["Cor"], s)
    return best

In [50]:
# ── Cell 6: train ────────────────────────────────────────────────────────────
train_rows, dev_rows = [], []
for lang in LANGS:
    tr, dv = split_dev(load(lang, "train"))
    train_rows += tr; dev_rows += dv
print(f"train={len(train_rows)}  dev={len(dev_rows)}")

model = SpanTagger().to(DEVICE)

# XLM-R-large's embedding matrix is 256M of its 560M parameters (250k vocab).
# Freezing it drops its gradient and both Adam states — about 3 GB — at little
# cost, since the multilingual embeddings are already well trained.
model.encoder.embeddings.requires_grad_(False)

train_loader = DataLoader(SpanData(train_rows), batch_size=BATCH, shuffle=True,
                          collate_fn=collate)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, weight_decay=0.01, foreach=False)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(0.1 * len(train_loader) * EPOCHS), len(train_loader) * EPOCHS)

for epoch in range(EPOCHS):
    loss = run_epoch(model, train_loader, optimizer, scheduler)
    print(f"epoch {epoch + 1}: loss {loss:.4f}", flush=True)
    # checkpoint every epoch so a disconnect costs one epoch, not the whole run
    torch.save(model.state_dict(), f"{OUT_DIR}/span_tagger.pt")

train=12085  dev=3017


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  step 50/1511  loss 1.7529
  step 100/1511  loss 1.5722
  step 150/1511  loss 1.3554
  step 200/1511  loss 1.2298
  step 250/1511  loss 1.1378
  step 300/1511  loss 1.0886
  step 350/1511  loss 1.0384
  step 400/1511  loss 1.0111
  step 450/1511  loss 0.9844
  step 500/1511  loss 0.9629
  step 550/1511  loss 0.9391
  step 600/1511  loss 0.9216
  step 650/1511  loss 0.9019
  step 700/1511  loss 0.8878
  step 750/1511  loss 0.8751
  step 800/1511  loss 0.8631
  step 850/1511  loss 0.8570
  step 900/1511  loss 0.8452
  step 950/1511  loss 0.8352
  step 1000/1511  loss 0.8286
  step 1050/1511  loss 0.8235
  step 1100/1511  loss 0.8152
  step 1150/1511  loss 0.8078
  step 1200/1511  loss 0.8025
  step 1250/1511  loss 0.7969
  step 1300/1511  loss 0.7905
  step 1350/1511  loss 0.7863
  step 1400/1511  loss 0.7814
  step 1450/1511  loss 0.7759
  step 1500/1511  loss 0.7714
epoch 1: loss 0.7713
  step 50/1511  loss 0.6043
  step 100/1511  loss 0.6043
  step 150/1511  loss 0.6059
  step 200/15

In [51]:
# ── Cell 7: evaluate per language ────────────────────────────────────────────
print(f"\n{'lang':<6}{'thr':>6}{'Cor':>9}{'Cor_lbl':>9}{'IoU':>9}   (mark_none Cor)")
thresholds = {}
for lang in LANGS:
    rows = [r for r in dev_rows if r["language"] == lang]
    cps = predict_char_probs(model, rows)
    thr, cor, scores = tune_threshold(rows, cps)
    thresholds[lang] = thr
    refs = [{"id": r["id"], "labels": r["labels"], "text_len": len(r["response"])}
            for r in rows]
    floor = evaluate(refs, [{"id": r["id"], "labels": []} for r in rows])["Cor"]
    print(f"{lang:<6}{thr:>6.2f}{scores['Cor']:>9.3f}{scores['Cor_lbl']:>9.3f}"
          f"{scores['IoU']:>9.3f}   {floor:.3f}")



lang     thr      Cor  Cor_lbl      IoU   (mark_none Cor)
en      0.25    0.364    0.313    0.319   0.253
fr      0.25    0.380    0.325    0.335   0.255
it      0.25    0.408    0.353    0.365   0.263
zh      0.35    0.436    0.407    0.408   0.365


In [52]:
# ── Cell 8: predict test and write submission ────────────────────────────────
for lang in LANGS:
    rows = load(lang, "test")
    cps = predict_char_probs(model, rows)
    preds = build_preds(rows, cps, thresholds[lang])
    path = f"{OUT_DIR}/predictions_{lang}.jsonl"
    with open(path, "w", encoding="utf-8") as fh:
        for p in preds:
            fh.write(json.dumps(p, ensure_ascii=False) + "\n")
    n_spans = sum(len(p["labels"]) for p in preds)
    n_empty = sum(1 for p in preds if not p["labels"])
    print(f"{lang}: {len(preds)} rows, {n_spans} spans, {n_empty} empty -> {path}")


en: 1201 rows, 3601 spans, 429 empty -> /kaggle/working/predictions_en.jsonl
fr: 1233 rows, 4769 spans, 401 empty -> /kaggle/working/predictions_fr.jsonl
it: 1254 rows, 2006 spans, 492 empty -> /kaggle/working/predictions_it.jsonl
zh: 1210 rows, 749 spans, 794 empty -> /kaggle/working/predictions_zh.jsonl


In [53]:
!wget -q https://a3s.fi/mickusti-2007780-pub/participant_kit.shroom_visions.zip -O /tmp/kit.zip
!unzip -oq /tmp/kit.zip -d /tmp/kit
!cd /kaggle/working && python /tmp/kit/participant_kit/format_checker.py predictions_en.jsonl predictions_fr.jsonl predictions_it.jsonl predictions_zh.jsonl --reference-dir /kaggle/working/distrib

Checked 4 file(s), 4898 row(s), 11125 span(s).
Languages: en, fr, it, zh
OK: submission format looks valid.


In [54]:
from IPython.display import FileLink
!cd /kaggle/working && zip -q predictions.zip predictions_en.jsonl predictions_fr.jsonl predictions_it.jsonl predictions_zh.jsonl
FileLink('predictions.zip')

/kaggle/working/predictions.zip

In [55]:
# ── Cell 9: category assignment strategies (no retraining needed) ────────────
#
# The span head stays exactly as trained. Only the way we turn per-token
# category scores into span labels changes. Compares three strategies on dev.

import numpy as np
import torch
from torch.utils.data import DataLoader


@torch.no_grad()
def predict_char_full(model, rows):
    """Per-character hallucination probability plus the full category distribution.

    Returns (char_prob, char_cat_probs) per row, where char_cat_probs has shape
    (n_chars, len(CATEGORIES)). Keeping the distribution instead of an argmax is
    what lets us pool categories over a span or a whole response.
    """
    model.eval()
    dataset = SpanData(rows, labeled=False)
    loader = DataLoader(dataset, batch_size=BATCH, shuffle=False, collate_fn=collate)
    results, cursor = [], 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        span_logit, cat_logit = model(batch["input_ids"], batch["attention_mask"],
                                      batch["keep"], batch["keep_mask"])
        probs = torch.sigmoid(span_logit).cpu().numpy()
        cat_p = torch.softmax(cat_logit, dim=-1).cpu().numpy()
        for i in range(len(probs)):
            row = rows[cursor]
            enc, keep = dataset.cache[cursor]
            cursor += 1
            offs = [enc["offset_mapping"][k] for k in keep]
            n = len(row["response"])
            cp = np.zeros(n, dtype=np.float32)
            cd = np.zeros((n, len(CATEGORIES)), dtype=np.float32)
            for j, (a, b) in enumerate(offs):
                e = min(b, n)
                cp[a:e] = probs[i][j]
                cd[a:e] = cat_p[i][j]
            results.append((cp, cd))
    return results


def spans_with_strategy(char_prob, char_dist, threshold, strategy):
    """Build spans, assigning categories by one of three strategies.

    per_token : argmax at each character; a span breaks where the label changes
    per_span  : one label per contiguous run, pooled over its characters
    per_resp  : one label for the whole response, pooled over flagged characters
    """
    n = len(char_prob)
    above = char_prob >= threshold
    if not above.any():
        return []

    resp_label = int(char_dist[above].sum(axis=0).argmax())

    # contiguous runs of flagged characters
    runs, i = [], 0
    while i < n:
        if not above[i]:
            i += 1
            continue
        j = i
        while j < n and above[j]:
            j += 1
        runs.append((i, j))
        i = j

    spans = []
    for a, b in runs:
        if strategy == "per_resp":
            pieces = [(a, b, resp_label)]
        elif strategy == "per_span":
            pieces = [(a, b, int(char_dist[a:b].sum(axis=0).argmax()))]
        else:  # per_token — split the run wherever the argmax label changes
            labels = char_dist[a:b].argmax(axis=1)
            pieces, s = [], 0
            for k in range(1, len(labels) + 1):
                if k == len(labels) or labels[k] != labels[s]:
                    pieces.append((a + s, a + k, int(labels[s])))
                    s = k
        for s, e, lab in pieces:
            spans.append({"start": int(s), "end": int(e),
                          "prob": float(round(float(char_prob[s:e].mean()), 6)),
                          "label": CATEGORIES[lab]})
    return spans


def eval_strategy(rows, char_preds, threshold, strategy):
    refs = [{"id": r["id"], "labels": r["labels"], "text_len": len(r["response"])}
            for r in rows]
    preds = [{"id": r["id"],
              "labels": spans_with_strategy(cp, cd, threshold, strategy)}
             for r, (cp, cd) in zip(rows, char_preds)]
    scores = evaluate(refs, preds)
    counts = [len({s["label"] for s in p["labels"]}) for p in preds if p["labels"]]
    scores["cats_per_resp"] = float(np.mean(counts)) if counts else 0.0
    scores["n_flagged"] = len(counts)
    return scores


# ── run the comparison ───────────────────────────────────────────────────────
STRATEGIES = ["per_token", "per_span", "per_resp"]
best_strategy, best_thr = {}, {}

print(f"{'lang':<5}{'strategy':<12}{'thr':>6}{'Cor':>9}{'Cor_lbl':>10}{'cats/resp':>11}")
print("-" * 53)
for lang in LANGS:
    rows = [r for r in dev_rows if r["language"] == lang]
    preds_full = predict_char_full(model, rows)
    best = (None, None, -1)
    for strategy in STRATEGIES:
        top = (None, -1, None)
        for t in np.arange(0.10, 0.91, 0.05):
            s = eval_strategy(rows, preds_full, float(t), strategy)
            # Cor+Lbl is listed first on the leaderboard, so tune for it
            if s["Cor_lbl"] > top[1]:
                top = (float(t), s["Cor_lbl"], s)
        thr, cl, s = top
        print(f"{lang:<5}{strategy:<12}{thr:>6.2f}{s['Cor']:>9.3f}"
              f"{s['Cor_lbl']:>10.3f}{s['cats_per_resp']:>11.2f}")
        if cl > best[2]:
            best = (strategy, thr, cl)
    best_strategy[lang], best_thr[lang] = best[0], best[1]
    print(f"{'':5}-> best: {best[0]} @ {best[1]:.2f}  (Cor_lbl {best[2]:.3f})")
    print()

print("chosen:", {l: (best_strategy[l], round(best_thr[l], 2)) for l in LANGS})

lang strategy       thr      Cor   Cor_lbl  cats/resp
-----------------------------------------------------
en   per_token     0.35    0.362     0.326       1.09
en   per_span      0.35    0.362     0.325       1.09
en   per_resp      0.35    0.362     0.324       1.00
     -> best: per_token @ 0.35  (Cor_lbl 0.326)

fr   per_token     0.35    0.367     0.326       1.14
fr   per_span      0.25    0.380     0.326       1.23
fr   per_resp      0.35    0.367     0.320       1.00
     -> best: per_token @ 0.35  (Cor_lbl 0.326)

it   per_token     0.30    0.406     0.360       1.12
it   per_span      0.30    0.406     0.361       1.11
it   per_resp      0.30    0.406     0.360       1.00
     -> best: per_span @ 0.30  (Cor_lbl 0.361)

zh   per_token     0.45    0.424     0.408       1.04
zh   per_span      0.45    0.424     0.408       1.04
zh   per_resp      0.35    0.436     0.407       1.00
     -> best: per_token @ 0.45  (Cor_lbl 0.408)

chosen: {'en': ('per_token', 0.35), 'fr': ('per_t

In [56]:
!ls /kaggle/working/shroom-vis-images | head -3
!ls /kaggle/working/shroom-vis-images | wc -l

10006154746_45ce02f34b_o.jpg
100784627_485946ea31_o.jpg
10082299166_6743afcc4b_o.jpg
ls: write error: Broken pipe
2495


In [59]:
# =============================================================================
#  SHROOM-visions 2026 — Day 4: visual grounding features
#
#  For every response token, score it twice under Qwen2-VL-2B with teacher
#  forcing: once WITH the image, once with the image REMOVED. The difference in
#  log-probability measures how much that token depended on seeing the image.
#
#  Removal, not Gaussian noise: Yin et al. (arXiv 2504.10020) show noise makes
#  the model randomly overlook parts of the image, so the gap reflects what the
#  noise destroyed rather than what was grounded. Full removal is deterministic.
#
#  Output: visual_feats_{split}_{lang}.npz  — per example, character offsets
#  plus six per-token features.
# =============================================================================

# ── Cell V1: setup ───────────────────────────────────────────────────────────
SMOKE_TEST = False
MAX_EXAMPLES = 300

import json, os, time, gc
import numpy as np
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

DISTRIB   = "/kaggle/working/distrib"
IMAGE_DIR = "/kaggle/working/shroom-vis-images"
OUT_DIR   = "/kaggle/working"
MODEL_ID  = "Qwen/Qwen2-VL-2B-Instruct"
LANGS     = ["en", "fr", "it", "zh"]

# Cap visual tokens. Qwen2-VL uses dynamic resolution, and an unconstrained
# photo can produce >1500 visual tokens — the difference between a 3-hour run
# and a 12-hour one.
MIN_PIXELS = 64 * 28 * 28
MAX_PIXELS = 256 * 28 * 28

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


# ── Cell V2: load model ──────────────────────────────────────────────────────
# 2B in fp16 is ~4.4 GB — no quantization needed on a 15 GB T4, and fp16
# inference is faster than 4-bit at this size.
try:
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID, dtype=torch.float16, device_map="auto")
except TypeError:  # older transformers spells it torch_dtype
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto")
model.eval()
processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
tokenizer = processor.tokenizer
print("model loaded")


# ── Cell V3: scoring ─────────────────────────────────────────────────────────
@torch.no_grad()
def score_response(image, prompt, response):
    """Teacher-force `response` twice and return per-token features.

    Returns (offsets, feats) where feats has one row per response token:
        [logp_img, ent_img, logp_noimg, ent_noimg, d_logp, d_ent]
    or (None, None) if the two tokenizations cannot be aligned.
    """
    # Character offsets come from tokenizing the response on its own.
    resp_enc = tokenizer(response, return_offsets_mapping=True,
                         add_special_tokens=False)
    resp_ids = resp_enc["input_ids"]
    offsets = resp_enc["offset_mapping"]
    if not resp_ids:
        return None, None

    def run(with_image):
        content = ([{"type": "image"}] if with_image else []) + \
                  [{"type": "text", "text": prompt}]
        messages = [{"role": "user", "content": content}]
        prefix = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        kwargs = {"text": [prefix + response], "return_tensors": "pt"}
        pre_kwargs = {"text": [prefix], "return_tensors": "pt"}
        if with_image:
            kwargs["images"] = [image]
            pre_kwargs["images"] = [image]
        inputs = processor(**kwargs).to(DEVICE)
        n_prefix = processor(**pre_kwargs)["input_ids"].shape[1]

        ids = inputs["input_ids"][0]
        # The response must tokenize identically in context, or offsets are wrong.
        if len(ids) - n_prefix != len(resp_ids):
            return None
        logits = model(**inputs).logits[0].float()
        # position i predicts token i+1
        step = logits[n_prefix - 1:-1]
        logprobs = torch.log_softmax(step, dim=-1)
        target = ids[n_prefix:]
        tok_logp = logprobs.gather(1, target.unsqueeze(1)).squeeze(1)
        entropy = -(logprobs.exp() * logprobs).sum(dim=-1)
        return tok_logp.cpu().numpy(), entropy.cpu().numpy()

    with_img = run(True)
    without_img = run(False)
    if with_img is None or without_img is None:
        return None, None

    logp_i, ent_i = with_img
    logp_n, ent_n = without_img
    feats = np.stack([logp_i, ent_i, logp_n, ent_n,
                      logp_i - logp_n, ent_i - ent_n], axis=1)
    return np.array(offsets, dtype=np.int32), feats.astype(np.float32)


# ── Cell V4: run over a split ────────────────────────────────────────────────
def load_rows(split, lang):
    kind = "labeled" if split == "train" else "unlabeled"
    with open(f"{DISTRIB}/shroom-vision.{split}.{lang}.{kind}.jsonl",
              encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]


def extract(split, lang, limit=None):
    rows = load_rows(split, lang)
    if limit:
        rows = rows[:limit]
    out_path = f"{OUT_DIR}/visual_feats_{split}_{lang}.npz"

    store, skipped, t0 = {}, 0, time.time()
    for i, row in enumerate(rows):
        img_path = os.path.join(IMAGE_DIR, row["image_name"])
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as exc:
            skipped += 1
            if skipped <= 3:
                print(f"  image unreadable {row['image_name']}: {exc}")
            continue
        offsets, feats = score_response(image, row["prompt"], row["response"])
        if feats is None:
            skipped += 1
            continue
        store[row["id"] + "|off"] = offsets
        store[row["id"] + "|f"] = feats
        if (i + 1) % 100 == 0:
            rate = (i + 1) / (time.time() - t0)
            eta = (len(rows) - i - 1) / rate / 60
            print(f"  {split}/{lang} {i + 1}/{len(rows)} "
                  f"{rate:.1f}/s  eta {eta:.0f} min  skipped {skipped}",
                  flush=True)

    np.savez_compressed(out_path, **store)
    done = len(store) // 2
    print(f"{split}/{lang}: {done} saved, {skipped} skipped, "
          f"{(time.time() - t0) / 60:.1f} min -> {out_path}", flush=True)
    return done, skipped

# ── Cell V5: run extraction ──────────────────────────────────────────────────
total = 0
for split in ["train"]:
    for lang in ["en"]:
        done, _ = extract(split, lang, MAX_EXAMPLES)
        total += done
        gc.collect(); torch.cuda.empty_cache()
print(f"\nall done: {total} examples")

device: cuda


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

model loaded


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  train/en 100/300 2.2/s  eta 2 min  skipped 0
  train/en 200/300 2.2/s  eta 1 min  skipped 0
  train/en 300/300 2.2/s  eta 0 min  skipped 0
train/en: 300 saved, 0 skipped, 2.3 min -> /kaggle/working/visual_feats_train_en.npz

all done: 300 examples


In [61]:
# ── Cell V6: does the signal exist? ──────────────────────────────────────────
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

d = np.load(f"{OUT_DIR}/visual_feats_train_en.npz")
rows = {json.loads(l)["id"]: json.loads(l)
        for l in open(f"{DISTRIB}/shroom-vision.train.en.labeled.jsonl")}

ys, gs, xs = [], [], []
for key in [k for k in d.files if k.endswith("|f")]:
    rid = key[:-2]
    row, offs, feats = rows[rid], d[rid + "|off"], d[key]
    gold = np.zeros(len(row["response"]))
    for s in row["labels"]:
        seg = gold[s["start"]:s["end"]]
        gold[s["start"]:s["end"]] = np.maximum(seg, s["prob"])
    for (a, b), f in zip(offs, feats):
        g = float(gold[a:b].max()) if b > a else 0.0
        ys.append(g > 0)        # any annotator marked it
        gs.append(g)            # graded probability, what Cor actually uses
        xs.append(f)

ys, gs, xs = np.array(ys), np.array(gs), np.array(xs)
names = ["logp_img", "ent_img", "logp_noimg", "ent_noimg", "d_logp", "d_ent"]
print(f"{ys.sum()} hallucinated / {len(ys)} tokens ({100*ys.mean():.1f}%)\n")
print(f"{'feature':<12}{'AUC':>8}{'Spearman':>11}")
for i, n in enumerate(names):
    auc = roc_auc_score(ys, -xs[:, i])
    rho = spearmanr(gs, -xs[:, i]).correlation
    print(f"{n:<12}{auc:>8.3f}{rho:>11.3f}")

3676 hallucinated / 35116 tokens (10.5%)

feature          AUC   Spearman
logp_img       0.499     -0.000
ent_img        0.479     -0.023
logp_noimg     0.503      0.004
ent_noimg      0.484     -0.018
d_logp         0.485     -0.016
d_ent          0.465     -0.037


In [62]:
# ── Cell V7: are the features actually aligned to their tokens? ──────────────
from collections import defaultdict
by_tok = defaultdict(list)
for key in [k for k in d.files if k.endswith("|f")]:
    rid = key[:-2]
    resp, offs, feats = rows[rid]["response"], d[rid + "|off"], d[key]
    for (a, b), f in zip(offs, feats):
        by_tok[resp[a:b]].append(f[0])          # logp_img

common = {t: np.mean(v) for t, v in by_tok.items() if len(v) >= 20}
order = sorted(common, key=common.get)
print("LOWEST logp (should be rare/contentful):", [repr(t) for t in order[:10]])
print("HIGHEST logp (should be common/function):", [repr(t) for t in order[-10:]])

LOWEST logp (should be rare/contentful): ["'.\\n\\n\\n\\n'", "' *'", "'Certainly'", "' –'", "'While'", "' You'", "'You'", "' Let'", "'Based'", "' While'"]
HIGHEST logp (should be common/function): ["'ing'", "'),'", "' of'", "' determine'", "'ation'", "'2'", "'s'", "' wall'", "'0'", '"\'t"']


# Visual Ablation Probe

Same two forward passes, different thing recorded. Instead of output log-probabilities we take **internal representations**, because the proxy-analyzer literature ([arXiv 2605.07209](https://arxiv.org/html/2605.07209)) gets cross-model transfer from activations, while our logit features sat at chance.

Per token, from three layers (early / middle / late), with and without the image:

| feature | question it asks |
|---|---|
| `cos(h_img, h_noimg)` | did the image rotate this token's representation? |
| `‖h_img − h_noimg‖` | by how much? |

Plus a fixed 32-dimensional random projection of the middle-layer difference, so a classifier can look for structure a single scalar would miss.

In [64]:
# ── Cell H1: hidden-state features ───────────────────────────────────────────

import numpy as np, torch, json, os, time, gc

PROJ_DIM = 32
_rng = np.random.RandomState(0)
_proj = None   # built lazily once the hidden size is known


@torch.no_grad()
def score_response_hidden(image, prompt, response):
    global _proj
    resp_enc = tokenizer(response, return_offsets_mapping=True,
                         add_special_tokens=False)
    resp_ids = resp_enc["input_ids"]
    offsets = resp_enc["offset_mapping"]
    if not resp_ids:
        return None, None
    n_resp = len(resp_ids)

    def run(with_image):
        content = ([{"type": "image"}] if with_image else []) + \
                  [{"type": "text", "text": prompt}]
        messages = [{"role": "user", "content": content}]
        prefix = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        kwargs = {"text": [prefix + response], "return_tensors": "pt"}
        if with_image:
            kwargs["images"] = [image]
        inputs = processor(**kwargs).to(DEVICE)
        out = model(**inputs, output_hidden_states=True)
        hs = out.hidden_states
        n = len(hs)
        picks = [n // 4, n // 2, (3 * n) // 4]
        # response tokens are the final n_resp positions in both passes
        return [hs[p][0, -n_resp:, :].float().cpu().numpy() for p in picks]

    a = run(True)
    b = run(False)
    if a[0].shape[0] != n_resp or b[0].shape[0] != n_resp:
        return None, None

    if _proj is None:
        _proj = _rng.randn(a[1].shape[1], PROJ_DIM).astype(np.float32) / np.sqrt(PROJ_DIM)

    cols = []
    for hi, hn in zip(a, b):
        num = (hi * hn).sum(axis=1)
        den = np.linalg.norm(hi, axis=1) * np.linalg.norm(hn, axis=1) + 1e-8
        cols.append(num / den)                        # cosine
        cols.append(np.linalg.norm(hi - hn, axis=1))  # distance
    feats = np.stack(cols, axis=1)
    proj = (a[1] - b[1]) @ _proj
    return np.array(offsets, dtype=np.int32), \
        np.concatenate([feats, proj], axis=1).astype(np.float32)

In [68]:
# ── Cell H2: extract ─────────────────────────────────────────────────────────
def extract_hidden(split, lang, limit=None):
    rows = load_rows(split, lang)
    if limit:
        rows = rows[:limit]
    store, skipped, t0 = {}, 0, time.time()
    for i, row in enumerate(rows):
        try:
            image = Image.open(os.path.join(IMAGE_DIR, row["image_name"])).convert("RGB")
        except Exception:
            skipped += 1
            continue
        offsets, feats = score_response_hidden(image, row["prompt"], row["response"])
        if feats is None:
            skipped += 1
            continue
        store[row["id"] + "|off"] = offsets
        store[row["id"] + "|f"] = feats
        if (i + 1) % 100 == 0:
            rate = (i + 1) / (time.time() - t0)
            print(f"  {i + 1}/{len(rows)} {rate:.1f}/s skipped {skipped}", flush=True)
    path = f"{OUT_DIR}/hidden_feats_{split}_{lang}.npz"
    np.savez_compressed(path, **store)
    print(f"{len(store) // 2} saved, {skipped} skipped, "
          f"{(time.time() - t0) / 60:.1f} min -> {path}", flush=True)


for split in ["test", "train"]:
    for lang in ["en", "fr", "it", "zh"]:
        print(f"=== {split}/{lang} ===", flush=True)
        extract_hidden(split, lang, None)
        gc.collect(); torch.cuda.empty_cache()

=== test/en ===
  100/1201 2.3/s skipped 0
  200/1201 2.3/s skipped 0
  300/1201 2.3/s skipped 0
  400/1201 2.2/s skipped 0
  500/1201 2.2/s skipped 0
  600/1201 2.2/s skipped 0
  700/1201 2.2/s skipped 0
  800/1201 2.2/s skipped 0
  900/1201 2.3/s skipped 0
  1000/1201 2.3/s skipped 0
  1100/1201 2.3/s skipped 0
  1200/1201 2.3/s skipped 0
1201 saved, 0 skipped, 8.6 min -> /kaggle/working/hidden_feats_test_en.npz
=== test/fr ===
  100/1233 2.4/s skipped 0
  200/1233 2.5/s skipped 0
  300/1233 2.5/s skipped 0
  400/1233 2.4/s skipped 0
  500/1233 2.3/s skipped 0
  600/1233 2.3/s skipped 0
  700/1233 2.3/s skipped 0
  800/1233 2.3/s skipped 0
  900/1233 2.3/s skipped 0
  1000/1233 2.3/s skipped 0
  1100/1233 2.4/s skipped 0
  1200/1233 2.4/s skipped 0
1233 saved, 0 skipped, 8.6 min -> /kaggle/working/hidden_feats_test_fr.npz
=== test/it ===
  100/1254 2.6/s skipped 0
  200/1254 2.6/s skipped 0
  300/1254 2.5/s skipped 0
  400/1254 2.4/s skipped 0
  500/1254 2.4/s skipped 0
  600/1254 2.

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  100/3799 2.4/s skipped 0
  200/3799 2.4/s skipped 0
  300/3799 2.4/s skipped 0
  400/3799 2.4/s skipped 0
  500/3799 2.4/s skipped 0
  600/3799 2.4/s skipped 0
  700/3799 2.4/s skipped 0
  800/3799 2.4/s skipped 0
  900/3799 2.4/s skipped 0
  1000/3799 2.5/s skipped 0
  1100/3799 2.5/s skipped 0
  1200/3799 2.5/s skipped 0
  1300/3799 2.5/s skipped 0
  1400/3799 2.5/s skipped 0
  1500/3799 2.5/s skipped 0
  1600/3799 2.4/s skipped 0
  1700/3799 2.4/s skipped 0
  1800/3799 2.4/s skipped 0
  1900/3799 2.4/s skipped 0
  2000/3799 2.4/s skipped 0
  2100/3799 2.4/s skipped 0
  2200/3799 2.3/s skipped 0
  2300/3799 2.3/s skipped 0
  2400/3799 2.3/s skipped 0
  2500/3799 2.3/s skipped 0
  2600/3799 2.3/s skipped 0
  2700/3799 2.3/s skipped 0
  2800/3799 2.3/s skipped 0
  2900/3799 2.3/s skipped 0
  3000/3799 2.3/s skipped 0
  3100/3799 2.3/s skipped 0
  3200/3799 2.3/s skipped 0
  3300/3799 2.3/s skipped 0
  3400/3799 2.3/s skipped 0
  3500/3799 2.3/s skipped 0
  3600/3799 2.3/s skipped 0
 

In [66]:
# ── Cell H3: is there signal? ────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr

d = np.load(f"{OUT_DIR}/hidden_feats_train_en.npz")
rows = {json.loads(l)["id"]: json.loads(l)
        for l in open(f"{DISTRIB}/shroom-vision.train.en.labeled.jsonl")}

ys, gs, xs = [], [], []
for key in [k for k in d.files if k.endswith("|f")]:
    rid = key[:-2]
    row, offs, feats = rows[rid], d[rid + "|off"], d[key]
    gold = np.zeros(len(row["response"]))
    for s in row["labels"]:
        seg = gold[s["start"]:s["end"]]
        gold[s["start"]:s["end"]] = np.maximum(seg, s["prob"])
    for (a, b), f in zip(offs, feats):
        g = float(gold[a:b].max()) if b > a else 0.0
        ys.append(g > 0); gs.append(g); xs.append(f)

ys, gs, xs = np.array(ys), np.array(gs), np.array(xs)
names = ["cos_early", "dist_early", "cos_mid", "dist_mid", "cos_late", "dist_late"]
print(f"{ys.sum()} hallucinated / {len(ys)} tokens ({100 * ys.mean():.1f}%)\n")
print(f"{'feature':<12}{'AUC':>8}{'Spearman':>11}")
for i, n in enumerate(names):
    # try both directions; a scalar can separate either way
    auc = roc_auc_score(ys, xs[:, i])
    print(f"{n:<12}{max(auc, 1 - auc):>8.3f}{spearmanr(gs, xs[:, i]).correlation:>11.3f}")

# can a classifier find structure in the full difference vector?
Xtr, Xte, ytr, yte = train_test_split(xs, ys, test_size=0.3, random_state=0, stratify=ys)
mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit((Xtr - mu) / sd, ytr)
probe = roc_auc_score(yte, clf.predict_proba((Xte - mu) / sd)[:, 1])
print(f"\nlogistic probe on all {xs.shape[1]} features: AUC {probe:.3f}")
print("(0.50 = no signal; >0.60 means activations carry what logits did not)")

3676 hallucinated / 35116 tokens (10.5%)

feature          AUC   Spearman
cos_early      0.528     -0.032
dist_early     0.544      0.049
cos_mid        0.537     -0.042
dist_mid       0.559      0.065
cos_late       0.503      0.001
dist_late      0.511     -0.010

logistic probe on all 38 features: AUC 0.677
(0.50 = no signal; >0.60 means activations carry what logits did not)


In [67]:
# ── Cell H4: same probe, split by response instead of by token ───────────────
from sklearn.model_selection import GroupShuffleSplit

groups = []
ys2, xs2 = [], []
for key in [k for k in d.files if k.endswith("|f")]:
    rid = key[:-2]
    row, offs, feats = rows[rid], d[rid + "|off"], d[key]
    gold = np.zeros(len(row["response"]))
    for s in row["labels"]:
        seg = gold[s["start"]:s["end"]]
        gold[s["start"]:s["end"]] = np.maximum(seg, s["prob"])
    for (a, b), f in zip(offs, feats):
        ys2.append((float(gold[a:b].max()) if b > a else 0.0) > 0)
        xs2.append(f); groups.append(rid)

ys2, xs2, groups = np.array(ys2), np.array(xs2), np.array(groups)
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
              .split(xs2, ys2, groups))
mu, sd = xs2[tr].mean(0), xs2[tr].std(0) + 1e-8
clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit((xs2[tr] - mu) / sd, ys2[tr])
auc = roc_auc_score(ys2[te], clf.predict_proba((xs2[te] - mu) / sd)[:, 1])
print(f"grouped by response: AUC {auc:.3f}   ({len(set(groups))} responses)")

# which half is doing the work — the 6 scalars or the 32 projection dims?
for label, cols in [("6 scalars only", slice(0, 6)), ("32 projection only", slice(6, 38))]:
    m, s = xs2[tr][:, cols].mean(0), xs2[tr][:, cols].std(0) + 1e-8
    c = LogisticRegression(max_iter=2000, class_weight="balanced")
    c.fit((xs2[tr][:, cols] - m) / s, ys2[tr])
    a = roc_auc_score(ys2[te], c.predict_proba((xs2[te][:, cols] - m) / s)[:, 1])
    print(f"  {label:<20} AUC {a:.3f}")

grouped by response: AUC 0.636   (300 responses)
  6 scalars only       AUC 0.630
  32 projection only   AUC 0.583
